In [2]:
import pandas as pd

conversations = pd.read_csv(
    "../data/phase3/conversations.csv"
)

session_preferences = pd.read_csv(
    "../data/phase3/session_preferences.csv"
)

print("Conversations shape:", conversations.shape)
print("Session preferences shape:", session_preferences.shape)

print("\nConversation columns:")
print(conversations.columns.tolist())

print("\nSession preference columns:")
print(session_preferences.columns.tolist())

Conversations shape: (29666, 12)
Session preferences shape: (10000, 10)

Conversation columns:
['conversation_id', 'session_id', 'customer_id', 'turn_number', 'speaker', 'message', 'intent', 'category', 'subcategory', 'health_preference', 'budget', 'purpose']

Session preference columns:
['session_id', 'customer_id', 'session_start', 'intent', 'category', 'subcategory', 'health_preference', 'budget', 'purpose', 'confidence']


In [3]:
print("===== SAMPLE CONVERSATIONS =====")
print(
    conversations[
        [
            "conversation_id",
            "session_id",
            "customer_id",
            "turn_number",
            "speaker",
            "message",
            "intent",
            "category",
            "subcategory",
            "health_preference",
            "budget",
            "purpose"
        ]
    ].head(10).to_string(index=False)
)

print("\n===== SAMPLE SESSION PREFERENCES =====")
print(
    session_preferences.head(10).to_string(index=False)
)

===== SAMPLE CONVERSATIONS =====
conversation_id session_id customer_id  turn_number   speaker                                                                             message                 intent                 category        subcategory health_preference  budget          purpose
     CON0000001    S000001      C01031            1  customer                I am looking for foodgrains, oil & masala that is healthy under ₹150 product_recommendation Foodgrains, Oil & Masala Edible Oils & Ghee           healthy   150.0              NaN
     CON0000002    S000002      C00486            1  customer                                        I am looking for eggs, meat & fish under ₹50 product_recommendation        Eggs, Meat & Fish Pork & Other Meats              none    50.0              NaN
     CON0000003    S000003      C01729            1  customer                                I am looking for beauty & hygiene that is high fiber product_recommendation         Beauty & Hygiene      

In [4]:
# Create clean session context

session_context = session_preferences[
    [
        "session_id",
        "customer_id",
        "intent",
        "category",
        "subcategory",
        "health_preference",
        "budget",
        "purpose",
        "confidence"
    ]
].copy()

# Fill missing values
session_context = session_context.fillna("unknown")

print("Session context shape:", session_context.shape)
print("\nSample session contexts:")
print(session_context.head(10).to_string(index=False))

Session context shape: (10000, 9)

Sample session contexts:
session_id customer_id                 intent                 category        subcategory health_preference  budget          purpose  confidence
   S000001      C01031 product_recommendation Foodgrains, Oil & Masala Edible Oils & Ghee           healthy     150     personal_use        0.78
   S000002      C00486 product_recommendation        Eggs, Meat & Fish Pork & Other Meats              none      50 meal_preparation        0.93
   S000003      C01729 product_recommendation         Beauty & Hygiene          Skin Care        high_fiber     100 meal_preparation        0.98
   S000004      C01931 product_recommendation     Gourmet & World Food            unknown              none     100       family_use        0.97
   S000005      C00730 product_recommendation    Bakery, Cakes & Dairy            unknown           healthy     300      quick_snack        0.84
   S000006      C02722 product_recommendation    Bakery, Cakes & Dairy

In [5]:
master_products = pd.read_csv("../data/phase1/master_products.csv")
print("Master product columns:")
print(master_products.columns.tolist())

print("\nsample products:")
print(master_products.head().to_string(index=False))

Master product columns:
['product_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'market_price', 'product_type', 'rating', 'description', 'serving_basis_g', 'calories_kcal', 'protein_g', 'carbohydrates_g', 'sugar_g', 'fat_g', 'saturated_fat_g', 'fiber_g', 'sodium_mg', 'vegetarian', 'vegan', 'gluten_free', 'contains_milk', 'contains_nuts', 'contains_soy', 'health_tags']

sample products:
product_id                                            product_name               category           subcategory            brand  price  market_price             product_type  rating                                                                                                                                                                                                                                                                                                                                                                                                                         

In [6]:
product_docs = master_products[
    [
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "price",
        "market_price",
        "product_type",
        "rating",
        "description",
        "calories_kcal",
        "protein_g",
        "carbohydrates_g",
        "sugar_g",
        "fat_g",
        "fiber_g",
        "sodium_mg",
        "vegetarian",
        "vegan",
        "gluten_free",
        "contains_milk",
        "contains_nuts",
        "contains_soy",
        "health_tags"
    ]
].copy()

print("Product documents shape:", product_docs.shape)
print(product_docs.head(2).to_string(index=False))

Product documents shape: (27555, 24)
product_id                           product_name               category           subcategory            brand  price  market_price           product_type  rating                                                                                                                                                                                                                                                                                                                        description  calories_kcal  protein_g  carbohydrates_g  sugar_g  fat_g  fiber_g  sodium_mg  vegetarian  vegan  gluten_free  contains_milk  contains_nuts  contains_soy health_tags
   P000001 Garlic Oil - Vegetarian Capsule 500 mg       Beauty & Hygiene             Hair Care Sri Sri Ayurveda  220.0         220.0       Hair Oil & Serum     4.1                                                                                                          This Product contains Garlic Oil that is

In [7]:
def create_product_document(row):
    return f"""
Product ID: {row['product_id']}
Product Name: {row['product_name']}
Brand: {row['brand']}
Category: {row['category']}
Subcategory: {row['subcategory']}
Product Type: {row['product_type']}

Price: ₹{row['price']}
Market Price: ₹{row['market_price']}
Rating: {row['rating']}

Description: {row['description']}

Nutrition per 100g:
Calories: {row['calories_kcal']} kcal
Protein: {row['protein_g']} g
Carbohydrates: {row['carbohydrates_g']} g
Sugar: {row['sugar_g']} g
Fat: {row['fat_g']} g
Fiber: {row['fiber_g']} g
Sodium: {row['sodium_mg']} mg

Attributes:
Vegetarian: {row['vegetarian']}
Vegan: {row['vegan']}
Gluten Free: {row['gluten_free']}
Contains Milk: {row['contains_milk']}
Contains Nuts: {row['contains_nuts']}
Contains Soy: {row['contains_soy']}

Health Tags: {row['health_tags']}
""".strip()


product_docs["document"] = product_docs.apply(
    create_product_document,
    axis=1
)

print(product_docs[["product_id", "product_name", "document"]].head(2).to_string(index=False))

product_id                           product_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                document
   P000001 Garlic Oil - Vegetarian Capsule 500 mg                             

In [9]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.Collecting sentence-transformers
  Using cached sentence_transformers-6.0.1-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-5.16.1-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached huggingface_hub-1.30.0-py3-none-any.whl.metadata (16 kB)
  Using cached torch-2.14.0-cp312-cp312-win_amd64.whl.metadata (38 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached regex-2026.9.3-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached typer-0.27.2-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified



In [1]:
import sys 
print(sys.executable)
print(sys.version)

c:\Users\Nilabha\anaconda3\python.exe
3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]
